# Scraping Twitter/X — Selenium WebDriver Engine
> **Portfolio Project:** Dynamic Browser Automation for Social Media OSINT  
> **Topic:** Geopolitical Intelligence — US–Iran WW3 Escalation Discourse  
> **Target Output:** `data/raw/`  
> **Keyword:** `("world war" OR ww3 OR wwiii) (iran OR tehran) (america OR us OR usa OR washington) lang:en -filter:retweets`

---
## Pipeline Workflow
1. [Environment & WebDriver Initialization](#1-persiapan--instalasi)
2. [Interactive Authentication & Session Cookie Management](#2-setup-webdriver--login)
3. [Twitter Advanced Search Traversal](#3-navigasi-twitter-advanced-search)
4. [Infinite Scroll & Virtual DOM Extraction](#4-scraping-tweet-scroll--extract)
5. [Data Validation & Deduplication](#5-load--validasi-data)
6. [Export to Raw Storage](#6-simpan-ke-csv)


## 1. Persiapan & Instalasi

### Apa itu Selenium?
**Selenium WebDriver** adalah library Python yang mengontrol browser secara otomatis — klik, scroll, isi form, dan ekstrak konten halaman web. Untuk scraping Twitter, Selenium membuka Chrome/Firefox, login ke Twitter, lalu navigasi ke halaman Advanced Search.

### Persyaratan
| Komponen | Versi | Link |
|---|---|---|
| Python | ≥ 3.8 | [python.org](https://python.org) |
| selenium | ≥ 4.0 | `pip install selenium` |
| webdriver-manager | latest | `pip install webdriver-manager` |
| Chrome/Firefox | latest | Sudah ada di sistem |

### Kelebihan Selenium
✅ Dapat login dengan cookies browser asli  
✅ Bypass deteksi bot lebih mudah  
✅ Dukungan JavaScript rendering  
✅ Cocok untuk data yang membutuhkan scroll panjang

In [1]:
# ── Install Dependencies ──
import subprocess, sys

packages = [
    "selenium>=4.0.0",
    "webdriver-manager",
    "pandas",
    ]

print("📦 Menginstall dependencies...")
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    name = pkg.split(">=")[0].split("==")[0]
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {name}")

print()
print("✅ Semua package siap!")


📦 Menginstall dependencies...
  ✅ selenium
  ✅ webdriver-manager
  ✅ pandas
  ✅ openpyxl

✅ Semua package siap!


In [2]:
# ── Import Library ──
import time
import json
import re
import os
import pickle
from pathlib import Path
from datetime import datetime
from urllib.parse import quote

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, StaleElementReferenceException
)

try:
    from webdriver_manager.chrome import ChromeDriverManager
    WDM_AVAILABLE = True
except ImportError:
    WDM_AVAILABLE = False

print("✅ Semua library berhasil diimport")
print(f"   Selenium versi : {webdriver.__version__ if hasattr(webdriver, '__version__') else 'OK'}")


✅ Semua library berhasil diimport
   Selenium versi : 4.43.0


## 2. Setup WebDriver & Login

### Strategi Login
Ada dua cara login ke Twitter menggunakan Selenium:

**Metode A — Cookies (Rekomendasi):**  
Simpan cookies dari browser yang sudah login, load di Selenium. Lebih stabil dan tidak trigger verifikasi.

**Metode B — Username & Password:**  
Selenium isi form login otomatis. Risiko trigger CAPTCHA atau verifikasi 2FA.

In [ ]:
# ─── SEL KHUSUS AUTENTIKASI SCRAPING (SELENIUM) ───────────────────────
import os
import getpass
from pathlib import Path

# File session cookie
COOKIES_FILE = Path("data/raw/twitter_cookies.pkl")

# Ambil kredensial dari environment variable atau input interaktif aman
TWITTER_USERNAME = os.environ.get("TWITTER_USERNAME")
TWITTER_PASSWORD = os.environ.get("TWITTER_PASSWORD")

if not TWITTER_USERNAME:
    TWITTER_USERNAME = input("Masukkan Username/Email Twitter (Enter jika login via cookies): ").strip()
if TWITTER_USERNAME and not TWITTER_PASSWORD:
    TWITTER_PASSWORD = getpass.getpass("Masukkan Password Twitter (tidak tampil di layar): ").strip()

print("=" * 55)
print("  STATUS AUTENTIKASI SELENIUM")
print("=" * 55)
print(f"  Username    : {TWITTER_USERNAME if TWITTER_USERNAME else '(Menggunakan Session Cookies / Manual)'}")
print(f"  Cookies File: {COOKIES_FILE} ({'Ada' if COOKIES_FILE.exists() else 'Belum ada'})")
print("=" * 55)


  KONFIGURASI LOGIN
  Username    : HororPotong
  Password    : ***************
  Cookies     : twitter_cookies.pkl (belum ada)

  Urutan login:
  1. Cek cookies → jika ada, load cookies (skip login)
  2. Jika tidak ada → login dengan username/password
  3. Simpan cookies untuk sesi berikutnya


In [4]:
# ── Inisialisasi Chrome WebDriver ──

def init_driver(headless: bool = False) -> webdriver.Chrome:
    """
    Inisialisasi Chrome WebDriver dengan konfigurasi anti-deteksi.
    
    Args:
        headless: True = Chrome tidak tampil (mode background)
                  False = Chrome tampil (lebih mudah untuk debug)
    
    Returns:
        WebDriver instance
    """
    options = Options()

    if headless:
        options.add_argument("--headless=new")         # Headless mode Chrome baru

    # ─── Konfigurasi Anti-Deteksi Bot ─────────────────────────
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--window-size=1366,768")
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    )
    # ──────────────────────────────────────────────────────────

    # Auto-install ChromeDriver yang sesuai versi Chrome
    if WDM_AVAILABLE:
        service = Service(ChromeDriverManager().install())
    else:
        service = Service()  # Gunakan chromedriver dari PATH

    driver = webdriver.Chrome(service=service, options=options)

    # Sembunyikan tanda WebDriver dari JavaScript
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )

    print(f"✅ Chrome WebDriver siap (headless={headless})")
    return driver


# Test inisialisasi
# driver = init_driver(headless=False)
print("✅ Fungsi init_driver() siap. Uncomment baris terakhir untuk test.")


✅ Fungsi init_driver() siap. Uncomment baris terakhir untuk test.


In [5]:
# ── Login Metode A: Gunakan Cookies ──

def save_cookies(driver: webdriver.Chrome, filepath: Path):
    """Simpan cookies sesi Twitter ke file."""
    pickle.dump(driver.get_cookies(), open(filepath, "wb"))
    print(f"✅ Cookies tersimpan: {filepath}")


def load_cookies(driver: webdriver.Chrome, filepath: Path) -> bool:
    """
    Load cookies dari file dan inject ke driver.
    Returns True jika berhasil.
    """
    if not filepath.exists():
        print(f"⚠️  File cookies tidak ditemukan: {filepath}")
        return False

    try:
        driver.get("https://twitter.com")
        time.sleep(2)
        cookies = pickle.load(open(filepath, "rb"))
        for cookie in cookies:
            try:
                driver.add_cookie(cookie)
            except Exception:
                pass  # Skip cookie yang tidak kompatibel
        driver.refresh()
        time.sleep(3)
        print(f"✅ Cookies berhasil diload dari: {filepath}")
        return True
    except Exception as e:
        print(f"❌ Gagal load cookies: {e}")
        return False


# ── Login Metode B: Username & Password ──

def login_with_credentials(driver: webdriver.Chrome, username: str, password: str) -> bool:
    """
    Login ke Twitter menggunakan username dan password.
    
    Returns True jika berhasil login.
    """
    wait = WebDriverWait(driver, 20)
    
    try:
        print("🌐 Membuka halaman login Twitter...")
        driver.get("https://twitter.com/login")
        time.sleep(3)

        # ─── Isi username ───────────────────────────────────────
        print("  Mengisi username...")
        username_field = wait.until(
            EC.presence_of_element_located((By.NAME, "text"))
        )
        username_field.clear()
        for char in username:          # Ketik karakter per karakter (mirip manusia)
            username_field.send_keys(char)
            time.sleep(0.05)
        time.sleep(1)
        username_field.send_keys(Keys.RETURN)
        time.sleep(2)

        # ─── Cek jika ada verifikasi email/phone ───────────────
        try:
            verify_field = driver.find_element(By.NAME, "text")
            print("  ⚠️  Twitter minta verifikasi email/phone — isi manual!")
            # verify_field.send_keys("email_atau_phone_kamu")
            # verify_field.send_keys(Keys.RETURN)
        except NoSuchElementException:
            pass

        # ─── Isi password ───────────────────────────────────────
        print("  Mengisi password...")
        password_field = wait.until(
            EC.presence_of_element_located((By.NAME, "password"))
        )
        for char in password:
            password_field.send_keys(char)
            time.sleep(0.05)
        time.sleep(1)
        password_field.send_keys(Keys.RETURN)
        time.sleep(5)

        # ─── Verifikasi login berhasil ──────────────────────────
        if "home" in driver.current_url or "twitter.com" in driver.current_url:
            print("✅ Login berhasil!")
            return True
        else:
            print(f"❌ Login gagal. URL saat ini: {driver.current_url}")
            return False

    except TimeoutException:
        print("❌ Timeout — halaman login tidak muncul")
        return False
    except Exception as e:
        print(f"❌ Error saat login: {e}")
        return False


print("✅ Fungsi login siap (save_cookies, load_cookies, login_with_credentials)")


✅ Fungsi login siap (save_cookies, load_cookies, login_with_credentials)


In [15]:
# ── MENJALANKAN LOGIN ──

# Pilih salah satu metode (uncomment yang ingin digunakan):

# ─── Opsi 1: Login dengan cookies yang sudah ada ───────────────
driver = init_driver(headless=False)
if not load_cookies(driver, COOKIES_FILE):
    # Fallback ke username/password jika cookies gagal
    login_with_credentials(driver, TWITTER_USERNAME, TWITTER_PASSWORD)
    save_cookies(driver, COOKIES_FILE)

# ─── Opsi 2: Login dengan username/password ─────────────────────
#driver = init_driver(headless=False)
#success = login_with_credentials(driver, TWITTER_USERNAME, TWITTER_PASSWORD)
#if success:
#    save_cookies(driver, COOKIES_FILE)   # Simpan untuk sesi berikutnya

print("📌 Uncomment blok kode di atas untuk menjalankan login.")
print()
print("Tips:")
print("  - Pertama kali: gunakan Opsi 2 (username/password)")
print("  - Selanjutnya: gunakan Opsi 1 (cookies — lebih cepat & aman)")


✅ Chrome WebDriver siap (headless=False)
⚠️  File cookies tidak ditemukan: twitter_cookies.pkl
🌐 Membuka halaman login Twitter...
  Mengisi username...
  ⚠️  Twitter minta verifikasi email/phone — isi manual!
  Mengisi password...
❌ Timeout — halaman login tidak muncul
✅ Cookies tersimpan: twitter_cookies.pkl
📌 Uncomment blok kode di atas untuk menjalankan login.

Tips:
  - Pertama kali: gunakan Opsi 2 (username/password)
  - Selanjutnya: gunakan Opsi 1 (cookies — lebih cepat & aman)


## 3. Navigasi Twitter Advanced Search

In [7]:
# ── Navigasi ke Twitter Advanced Search ──

def build_search_url(keyword: str) -> str:
    """
    Bangun URL Twitter Advanced Search dari keyword.
    
    Twitter Advanced Search URL format:
    https://twitter.com/search?q=ENCODED_QUERY&f=live&src=typed_query
    
    Parameter:
        f=live    → hasil real-time (bukan top)
        f=top     → hasil top/populer
        src=typed_query → sumber pencarian
    """
    encoded = quote(keyword)
    url = f"https://twitter.com/search?q={encoded}&f=live&src=typed_query"
    return url


KEYWORD   = '("world war" OR ww3 OR wwiii) (iran OR tehran) (america OR us OR usa OR washington) lang:en -filter:retweets'
SEARCH_URL = build_search_url(KEYWORD)

print(f"🔗 URL Advanced Search:")
print(f"   {SEARCH_URL[:100]}...")
print()


def navigate_to_search(driver: webdriver.Chrome, keyword: str, tab: str = "live") -> bool:
    """
    Navigasi ke hasil pencarian Twitter Advanced Search.
    
    Args:
        tab: 'live' = Latest, 'top' = Top tweets
    
    Returns True jika berhasil.
    """
    url = build_search_url(keyword)
    if tab == "top":
        url = url.replace("&f=live&", "&f=&")

    try:
        print(f"🌐 Navigasi ke Twitter Search...")
        driver.get(url)
        time.sleep(4)

        # Tunggu hingga tweet muncul
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        ))
        print(f"✅ Hasil pencarian muncul!")
        return True

    except TimeoutException:
        print("⚠️  Tidak ada hasil atau halaman lambat loading")
        print("   Cek: apakah sudah login? Apakah keyword terlalu spesifik?")
        return False

print("✅ Fungsi navigate_to_search() siap")


🔗 URL Advanced Search:
   https://twitter.com/search?q=%28%22world%20war%22%20OR%20ww3%20OR%20wwiii%29%20%28iran%20OR%20tehran...

✅ Fungsi navigate_to_search() siap


## 4. Scraping Tweet (Scroll & Extract)

In [8]:
# ── Ekstraksi Data dari Satu Tweet ──

def extract_tweet_data(article_element) -> dict:
    """
    Ekstrak data dari satu elemen artikel tweet.
    
    Mengembalikan dict dengan field:
        tweet_id, username, display_name, tweet, timestamp,
        likes, retweets, replies, quotes, views, url
    """
    data = {
        'tweet_id'     : '',
        'username'     : '',
        'display_name' : '',
        'tweet'        : '',
        'timestamp'    : '',
        'likes'        : 0,
        'retweets'     : 0,
        'replies'      : 0,
        'quotes'       : 0,
        'views'        : 0,
        'url'          : '',
    }

    try:
        # ─── Tweet ID & URL ─────────────────────────────────────
        try:
            link = article_element.find_element(
                By.CSS_SELECTOR, 'a[href*="/status/"]'
            )
            href = link.get_attribute('href')
            data['url'] = href
            # Ekstrak ID dari URL: https://twitter.com/user/status/12345
            match = re.search(r'/status/(\d+)', href)
            if match:
                data['tweet_id'] = match.group(1)
        except NoSuchElementException:
            pass

        # ─── Username ───────────────────────────────────────────
        try:
            user_el = article_element.find_element(
                By.CSS_SELECTOR, '[data-testid="User-Name"]'
            )
            spans = user_el.find_elements(By.TAG_NAME, 'span')
            texts = [s.text for s in spans if s.text.strip()]
            if texts:
                data['display_name'] = texts[0] if texts else ''
            # Username (@handle)
            user_link = user_el.find_element(By.CSS_SELECTOR, 'a[href*="/"]')
            handle = user_link.get_attribute('href').split('/')[-1]
            data['username'] = handle
        except (NoSuchElementException, IndexError):
            pass

        # ─── Timestamp ──────────────────────────────────────────
        try:
            time_el = article_element.find_element(By.TAG_NAME, 'time')
            data['timestamp'] = time_el.get_attribute('datetime')
        except NoSuchElementException:
            pass

        # ─── Teks Tweet ─────────────────────────────────────────
        try:
            tweet_el = article_element.find_element(
                By.CSS_SELECTOR, '[data-testid="tweetText"]'
            )
            data['tweet'] = tweet_el.text
        except NoSuchElementException:
            pass

        # ─── Engagement (likes, retweets, dll) ──────────────────
        def parse_count(text: str) -> int:
            """Parse '1.2K' → 1200, '3M' → 3000000, dsb."""
            text = text.replace(',', '').strip()
            if not text:
                return 0
            try:
                if text.endswith('K'):
                    return int(float(text[:-1]) * 1_000)
                elif text.endswith('M'):
                    return int(float(text[:-1]) * 1_000_000)
                else:
                    return int(text)
            except ValueError:
                return 0

        # Reply
        try:
            reply_el = article_element.find_element(
                By.CSS_SELECTOR, '[data-testid="reply"]'
            )
            data['replies'] = parse_count(reply_el.text)
        except NoSuchElementException:
            pass

        # Retweet
        try:
            rt_el = article_element.find_element(
                By.CSS_SELECTOR, '[data-testid="retweet"]'
            )
            data['retweets'] = parse_count(rt_el.text)
        except NoSuchElementException:
            pass

        # Like
        try:
            like_el = article_element.find_element(
                By.CSS_SELECTOR, '[data-testid="like"]'
            )
            data['likes'] = parse_count(like_el.text)
        except NoSuchElementException:
            pass


        # Views
        try:
            view_el = article_element.find_element(
                By.CSS_SELECTOR, 'a[href*="/analytics"]'
            )
            data['views'] = parse_count(view_el.text)
        except NoSuchElementException:
            pass
    except StaleElementReferenceException:
        pass  # Element sudah tidak di DOM — skip
    except Exception as e:
        pass

    return data

print("✅ Fungsi extract_tweet_data() siap")


✅ Fungsi extract_tweet_data() siap


In [16]:
# ── Scraping Utama: Scroll & Extract ──

def scrape_tweets(
    driver       : webdriver.Chrome,
    keyword      : str,
    max_tweets   : int  = 200,
    scroll_pause : float = 2.5,
    max_scrolls  : int  = 100,
) -> list[dict]:
    """
    Scroll halaman Twitter Search dan kumpulkan tweet.
    
    Args:
        max_tweets   : Jumlah tweet target
        scroll_pause : Jeda antar scroll (detik)
        max_scrolls  : Maksimal scroll (failsafe)
    
    Returns:
        List of tweet dicts
    """
    tweets_data = {}  # key: tweet_id, value: dict (otomatis deduplikasi)
    scroll_count = 0
    no_new_count = 0   # Counter jika tidak ada tweet baru
    MAX_NO_NEW   = 5   # Stop jika 5x scroll tidak ada tweet baru

    print(f"{'='*55}")
    print(f"  MEMULAI SCRAPING")
    print(f"{'='*55}")
    print(f"  Target   : {max_tweets} tweet")
    print(f"  Keyword  : {keyword[:50]}...")
    print()

    # Navigasi ke halaman pencarian
    if not navigate_to_search(driver, keyword):
        print("❌ Gagal navigasi ke halaman pencarian")
        return []

    while len(tweets_data) < max_tweets and scroll_count < max_scrolls:
        # ─── Ambil semua elemen tweet yang ada di DOM ────────────
        articles = driver.find_elements(
            By.CSS_SELECTOR, 'article[data-testid="tweet"]'
        )

        prev_count = len(tweets_data)

        for article in articles:
            tweet = extract_tweet_data(article)
            if tweet['tweet_id'] and tweet['tweet']:
                tweets_data[tweet['tweet_id']] = tweet

        new_count = len(tweets_data) - prev_count

        # ─── Progress report ─────────────────────────────────────
        scroll_count += 1
        print(
            f"  Scroll {scroll_count:>3} | "
            f"Tweet terkumpul: {len(tweets_data):>4} | "
            f"Baru: +{new_count}"
        )

        if new_count == 0:
            no_new_count += 1
            if no_new_count >= MAX_NO_NEW:
                print(f"  ⚠️  {MAX_NO_NEW}x scroll tanpa tweet baru — berhenti")
                break
        else:
            no_new_count = 0

        if len(tweets_data) >= max_tweets:
            print(f"  ✅ Target {max_tweets} tweet tercapai!")
            break

        # ─── Scroll ke bawah ─────────────────────────────────────
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(scroll_pause + (0.5 * (scroll_count % 3)))  # Jeda bervariasi

    result = list(tweets_data.values())[:max_tweets]
    print()
    print(f"✅ Total tweet berhasil dikumpulkan: {len(result):,}")
    return result


# ── JALANKAN SCRAPING ──────────────────────────────────────────────
# Uncomment kode di bawah setelah driver sudah login:

tweets_raw = scrape_tweets(
    driver      = driver,
    keyword     = KEYWORD,
    max_tweets  = 300,
    scroll_pause= 2.5,
)

print("✅ Fungsi scrape_tweets() siap")
print()
print("Jalankan setelah login:")
print("  tweets_raw = scrape_tweets(driver=driver, keyword=KEYWORD, max_tweets=300)")


  MEMULAI SCRAPING
  Target   : 300 tweet
  Keyword  : ("world war" OR ww3 OR wwiii) (iran OR tehran) (am...

🌐 Navigasi ke Twitter Search...
✅ Hasil pencarian muncul!
  Scroll   1 | Tweet terkumpul:    6 | Baru: +6
  Scroll   2 | Tweet terkumpul:   19 | Baru: +13
  Scroll   3 | Tweet terkumpul:   29 | Baru: +10
  Scroll   4 | Tweet terkumpul:   40 | Baru: +11
  Scroll   5 | Tweet terkumpul:   51 | Baru: +11
  Scroll   6 | Tweet terkumpul:   64 | Baru: +13
  Scroll   7 | Tweet terkumpul:   77 | Baru: +13
  Scroll   8 | Tweet terkumpul:   87 | Baru: +10
  Scroll   9 | Tweet terkumpul:  102 | Baru: +15
  Scroll  10 | Tweet terkumpul:  111 | Baru: +9
  Scroll  11 | Tweet terkumpul:  119 | Baru: +8
  Scroll  12 | Tweet terkumpul:  128 | Baru: +9
  Scroll  13 | Tweet terkumpul:  143 | Baru: +15
  Scroll  14 | Tweet terkumpul:  151 | Baru: +8
  Scroll  15 | Tweet terkumpul:  158 | Baru: +7
  Scroll  16 | Tweet terkumpul:  166 | Baru: +8
  Scroll  17 | Tweet terkumpul:  176 | Baru: +10
  Scro

## 5. Load & Validasi Data

In [19]:
# ── Konversi ke DataFrame & Validasi ──

def tweets_to_dataframe(tweets_raw: list) -> pd.DataFrame:
    """
    Konversi list tweet dicts ke DataFrame yang bersih.
    """
    if not tweets_raw:
        print("⚠️  Data kosong!")
        return pd.DataFrame()

    df = pd.DataFrame(tweets_raw)

    # ─── Konversi tipe data ──────────────────────────────────────
    numeric_cols = ['likes', 'retweets', 'replies', 'quotes', 'views']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    # ─── Timestamp ──────────────────────────────────────────────
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
        df['date'] = df['timestamp'].dt.date
        df['hour'] = df['timestamp'].dt.hour

    # ─── Hapus duplikat ──────────────────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=['tweet_id'])
    df = df.dropna(subset=['tweet'])
    df = df[df['tweet'].str.strip() != '']
    after = len(df)

    # ─── Tambah kolom metadata ───────────────────────────────────
    df['keyword']    = KEYWORD
    df['scraper']    = 'selenium'
    df['scraped_at'] = datetime.now().isoformat()

    # ─── Sort by timestamp ───────────────────────────────────────
    if 'timestamp' in df.columns:
        df = df.sort_values('timestamp', ascending=False).reset_index(drop=True)

    print(f"✅ DataFrame siap: {after:,} tweet ({before-after} duplikat dihapus)")
    return df


# ─── Jalankan setelah scraping ──────────────────────────────────
df = tweets_to_dataframe(tweets_raw)
print(df[['username','tweet','likes','retweets','timestamp']].head(5))


✅ DataFrame siap: 300 tweet (0 duplikat dihapus)
          username                                              tweet  likes  \
0         TheRam74  In part:\nYes Trump would be in prison.\nThe S...      0   
1    Publius_Philo  What else is new. Iran continues to play Trump...      0   
2    worldvidenews  BREAKING The Chilling Warnings World War Three...      0   
3       Irelandas1  Iran responds to US ceasefire proposal but Tru...      1   
4  Gravesweep43234  On principle, yes. In practice, USA will nuke ...      0   

   retweets                 timestamp  
0         0 2026-05-10 22:53:35+00:00  
1         0 2026-05-10 22:40:35+00:00  
2         0 2026-05-10 21:35:32+00:00  
3         1 2026-05-10 20:47:06+00:00  
4         0 2026-05-10 20:41:34+00:00  


## 6. Simpan ke CSV

In [23]:
# ── Simpan Dataset ──

def save_dataset(df: pd.DataFrame, prefix: str = "dataset_WW3_selenium"):
    """
    Simpan DataFrame ke CSV.
    Menambahkan timestamp pada nama file.
    """
    if df is None or len(df) == 0:
        print("⚠️  DataFrame kosong — tidak ada yang disimpan")
        return

    OUTPUT_DIR = Path("data/raw")
    OUTPUT_DIR.mkdir(exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d_%H%M")

    # ─── CSV ────────────────────────────────────────────────────
    csv_path = OUTPUT_DIR / f"{prefix}_{ts}.csv"
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"✅ CSV tersimpan   : {csv_path}")

    return csv_path


# Jalankan setelah df tersedia:
save_dataset(df)


✅ CSV tersimpan   : output_selenium\dataset_WW3_selenium_20260511_0854.csv


WindowsPath('output_selenium/dataset_WW3_selenium_20260511_0854.csv')